# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from rich.pretty import pprint

import numpy as np

from enderleaf.image import (
    to_pil,
    safe_pil_resize,
    merge_images,
    ImageMergeMode,
    get_circles,
    crop_image,
    Rectangle,
)
from enderleaf.draw import image_grid, concat_tile_resize
from enderleaf.enderleaf_ui import ui_show, controller, ui_main, ui_sidebar
from enderleaf.enderleaf_ctrl import CameraState
from enderscope.enderlights_pi import CardPoint, LIGHTS_CYCLE, LEN_LIGHTS_CYCLE
from enderleaf.tools import ensure_folder, format_datetime, write_dataframe

import panel as pn

## Initialize Preview

In [ ]:
ui_show().servable()

In [ ]:
controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
controller.acq_light_configurations

In [ ]:
controller.launch_acquisition(precise_focusing=False, switch_state=True)

In [ ]:
controller.acq_light_configurations = [LIGHTS_CYCLE[0]]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_no_light, *_ = controller.acquire_leaf_disc(switch_state=True)

controller.acq_light_configurations = [LIGHTS_CYCLE[1]]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_full_light, *_ = controller.acquire_leaf_disc(switch_state=True)

controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_nwse, *_ = controller.acquire_leaf_disc(switch_state=True)

controller.acq_light_configurations = [
    LIGHTS_CYCLE[6],
    LIGHTS_CYCLE[7],
    LIGHTS_CYCLE[8],
    LIGHTS_CYCLE[9],
]
pprint(controller.acq_light_configurations)
controller.set_lights(controller.acq_light_configurations[0], wait=1)
images_lr, *_ = controller.acquire_leaf_disc(switch_state=True)

# controller.acq_light_configurations = [
#     LIGHTS_CYCLE[10],
#     LIGHTS_CYCLE[11],
#     LIGHTS_CYCLE[12],
#     LIGHTS_CYCLE[13],
# ]
# pprint(controller.acq_light_configurations)
# controller.set_lights(controller.acq_light_configurations[0], wait=1)
# images_one_off, *_ = controller.acquire_leaf_disc(switch_state=True)

accu, cx, cy, r = get_circles(images_full_light[0], color_space="hsv", channel="s")["accepted"][0]
crop_data = Rectangle.from_circle((cx,cy,r+16))

images_no_light = [crop_image(i, crop_data) for i in images_no_light]
images_full_light = [crop_image(i, crop_data) for i in images_full_light]
images_nwse = [crop_image(i, crop_data) for i in images_nwse]
images_lr = [crop_image(i, crop_data) for i in images_lr]
# images_one_off = [crop_image(i, crop_data) for i in images_one_off]

to_pil(
    concat_tile_resize(
        [
            [
                images_no_light[0],
                images_full_light[0],
                merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MIN),
                #merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MAX),
                #merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.AVG),
                #merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MEDIAN),
            ],
            [
                images_no_light[0],
                images_full_light[0],
                merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MIN),
                #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MAX),
                #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.AVG),
                #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MEDIAN),
            ],
            # [
            #     images_no_light[0],
            #     images_full_light[0],
            #     merge_images(image_list=images_one_off, merge_mode=ImageMergeMode.MIN),
            #     #merge_images(image_list=images_one_off, merge_mode=ImageMergeMode.MAX),
            #     #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.AVG),
            #     #merge_images(image_list=images_lr, merge_mode=ImageMergeMode.MEDIAN),
            # ]
        ]
    )
)

In [ ]:
to_pil(images_full_light[0])

In [ ]:
to_pil(merge_images(image_list=images_nwse, merge_mode=ImageMergeMode.MIN))

In [ ]:
to_pil(images_nwse[0])

In [ ]:
to_pil(concat_tile_resize([images_nwse, images_lr, images_one_off]))

In [ ]:
controller.move_relative(0,0,1)

In [ ]:
def get_ligths_checkboxes(index: int):
    cps = [
        cp.value
        for cp in [CardPoint.NORTH, CardPoint.WEST, CardPoint.SOUTH, CardPoint.EAST]
    ]
    return pn.widgets.CheckBoxGroup(
        name=f"{index}",
        options=cps,
        value=[v.value for v in LIGHTS_CYCLE[min(index, LEN_LIGHTS_CYCLE - 1)]],
        inline=True,
    )

In [ ]:
controller.acq_light_configurations = [
    LIGHTS_CYCLE[2],
    LIGHTS_CYCLE[3],
    LIGHTS_CYCLE[4],
    LIGHTS_CYCLE[5],
]
controller.acq_light_configurations

In [ ]:
col = pn.Column( get_ligths_checkboxes(1))
col

In [ ]:
controls = {
    "AeEnable": False,
    "ExposureTime": 5000,
    "AnalogueGain": 1,
    "AwbEnable": False,
    "ColourGains": (2.3, 0.9),
}

# controls = {
#     "AeEnable": True,
#     "AwbEnable": True,
# }


controller.camera.set_controls(controls)

In [ ]:
col.append(get_ligths_checkboxes(len(col)))
len(col)

In [ ]:
controller.cycle_lights()
controller.top_lights.mean

In [ ]:
controller.set_top_lights(
    True,
    card_points=[
        CardPoint.NORTH,
        CardPoint.SOUTH,
        CardPoint.EAST,
        CardPoint.WEST,
    ],
)

In [ ]:
img_east, _ = controller.capture_array()

In [ ]:
to_pil(img_east)

In [ ]:
to_pil(img_west)

In [ ]:
import numpy as np

image_grid([img_east, img_west,np.minimum(img_east,img_west)], row_count=2)

In [ ]:
image_data = controller.last_job_data

sel_image = pn.widgets.IntSlider(
    name="Select image",
    start=0,
    end=len(image_data) - 1,
    value=0,
    sizing_mode="stretch_width",
)
ph_image = pn.pane.Placeholder()
json_data = pn.pane.JSON()


@pn.depends(sel_image.param.value, watch=True)
def on_index_changed(index):
    image = image_data[index][0]
    metadata = image_data[index][1]
    ph_image.object = safe_pil_resize(to_pil(image), 600, 600)
    json_data.object = {k: str(v) for k, v in metadata.items() if k != "image"}


on_index_changed(sel_image.value)

pn.Column(sel_image, pn.Row(ph_image, json_data))